<a href="https://colab.research.google.com/github/BassemRamdan/AI-Resume-Intelligence/blob/main/notebooks/01_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1: Dataset Inspection & EDA
This notebook focuses on thoroughly inspecting the raw Hugging Face resume dataset before any modeling or fine-tuning.

**Goals:**
- Download PDFs from Hugging Face.
- Inspect file validity (corrupted files, page counts, file sizes).
- Extract text using `pypdf` to identify text-based vs. image-based (scanned) PDFs.
- Detect exact duplicates via file hashing and near-duplicates via text hashing.
- Perform deep statistical analysis and visualization (word counts, category distributions, etc.).


In [ ]:
!pip install huggingface_hub pandas matplotlib seaborn scikit-learn pypdf tqdm datasets

In [ ]:
import os
import hashlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from huggingface_hub import snapshot_download
from pypdf import PdfReader
from tqdm.auto import tqdm

# Set plotting style
sns.set_theme(style="whitegrid")

# Download the dataset
print("Downloading dataset files from Hugging Face...")
dataset_path = snapshot_download(repo_id="BassemRamdan/data", repo_type="dataset")
print(f"Dataset downloaded to: {dataset_path}")

In [ ]:
def get_file_hash(filepath):
    """Generate MD5 hash for exact duplicate detection of files."""
    hasher = hashlib.md5()
    with open(filepath, 'rb') as afile:
        buf = afile.read()
        hasher.update(buf)
    return hasher.hexdigest()

data = []
categories = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d)) and not d.startswith('.git')]

print(f"Found {len(categories)} categories. Processing PDFs...")

for category in tqdm(categories, desc="Categories"):
    cat_path = os.path.join(dataset_path, category)
    for filename in os.listdir(cat_path):
        if filename.lower().endswith('.pdf'):
            file_path = os.path.join(cat_path, filename)
            
            # Metadata
            file_size_kb = os.path.getsize(file_path) / 1024
            file_hash = get_file_hash(file_path)
            
            text = ""
            num_pages = 0
            is_corrupted = False
            
            try:
                reader = PdfReader(file_path)
                num_pages = len(reader.pages)
                for page in reader.pages:
                    extracted = page.extract_text()
                    if extracted:
                        text += extracted + " "
            except Exception as e:
                is_corrupted = True
                
            clean_text = text.strip()
            
            data.append({
                "category": category,
                "filename": filename,
                "file_size_kb": file_size_kb,
                "num_pages": num_pages,
                "is_corrupted": is_corrupted,
                "file_hash": file_hash,
                "text": clean_text,
                "char_count": len(clean_text),
                "word_count": len(clean_text.split())
            })

df = pd.DataFrame(data)
print(f"\nDataset shape: {df.shape}")
df.head()

In [ ]:
print("--- PDF Validity Report ---")
corrupted_files = df[df['is_corrupted'] == True]
print(f"Number of corrupted PDFs (cannot be read by PyPDF): {len(corrupted_files)}")

empty_text_files = df[(df['is_corrupted'] == False) & (df['char_count'] == 0)]
print(f"Number of valid PDFs with NO extracted text: {len(empty_text_files)}")
print("*Note: These are likely scanned images requiring OCR in the next phase.")

In [ ]:
print("--- Duplicate Detection ---")
# Exact File Duplicates
exact_duplicates = df.duplicated(subset=['file_hash'], keep=False)
num_exact_duplicates = exact_duplicates.sum()
print(f"Number of exact duplicate files (identical binary): {num_exact_duplicates}")

if num_exact_duplicates > 0:
    display(df[exact_duplicates].sort_values('file_hash').head(4)[['category', 'filename', 'file_size_kb', 'file_hash']])

# Text-based Duplicates (Different files, same text content)
valid_text_df = df[df['char_count'] > 50]
text_duplicates = valid_text_df.duplicated(subset=['text'], keep=False)
num_text_duplicates = text_duplicates.sum()
print(f"\nNumber of text-based exact duplicates (different file, identical extracted text): {num_text_duplicates}")

In [ ]:
# 1. Class Distribution
plt.figure(figsize=(14, 8))
sns.countplot(data=df, y='category', order=df['category'].value_counts().index, palette="viridis")
plt.title('Resume Category Distribution', fontsize=16)
plt.xlabel('Number of Resumes', fontsize=12)
plt.ylabel('Category', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 2. PDF Page Count Distribution
plt.figure(figsize=(10, 5))
sns.histplot(df[df['num_pages'] > 0]['num_pages'], bins=20, color='blue', discrete=True)
plt.title('Distribution of PDF Page Counts', fontsize=14)
plt.xlabel('Number of Pages', fontsize=12)
plt.show()

In [ ]:
# 3. Word Count Distribution
plt.figure(figsize=(12, 6))
sns.histplot(df[df['word_count'] > 0]['word_count'], bins=50, color='green')
plt.title('Distribution of Resume Word Counts', fontsize=14)
plt.xlabel('Word Count', fontsize=12)
plt.show()

In [ ]:
# 4. Category vs Resume Length (Word Count)
plt.figure(figsize=(14, 10))
sns.boxplot(data=df[df['word_count'] > 0], x='word_count', y='category', palette="Set2")
plt.title('Word Count Distribution by Category', fontsize=16)
plt.xlabel('Word Count', fontsize=12)
plt.ylabel('Category', fontsize=12)
plt.tight_layout()
plt.show()